In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset,DataLoader
import torch.nn as nn
import torch.optim as optim 
import matplotlib.pyplot as plt

In [2]:
torch.manual_seed(42)

In [3]:
df = pd.read_csv('mnist_data.csv')
df.head()

,label,1x1,1x2,1x3,1x4,1x5,1x6,1x7,1x8,1x9,...,28x19,28x20,28x21,28x22,28x23,28x24,28x25,28x26,28x27,28x28
0,5,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,9,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [4]:
X = df.iloc[:,1:].values
y = df.iloc[:,0].values

In [5]:
X

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(60000, 784))

In [6]:
y

array([5, 0, 4, ..., 5, 6, 8], shape=(60000,))

In [7]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [8]:
X.shape

(60000, 784)

In [10]:
X_train.shape

(48000, 784)

In [11]:
X_test.shape

(12000, 784)

In [12]:
# scaling
X_train = X_train/255.0
X_test = X_test/255.0

In [13]:
# create custom dataset class
class CustomDataset(Dataset):
    def __init__(self,features,labels):
        self.features = torch.tensor(features,dtype=torch.float32).reshape(-1,1,28,28)
        self.labels = torch.tensor(labels,dtype=torch.long)
    def __len__(self):
        return len(self.features)
    def __getitem__(self,index):
        return self.features[index],self.labels[index]

In [14]:
# train and test dataset object
train_dataset = CustomDataset(X_train,y_train)

test_dataset = CustomDataset(X_test,y_test)

# create train and test loader

train_loader = DataLoader(train_dataset,batch_size=32,shuffle=True)
test_loader = DataLoader(test_dataset,batch_size=32,shuffle=False)

In [15]:
# define CNN class

class myCNN(nn.Module):
    def __init__(self,input_features):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(input_features,32,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(32),
            nn.MaxPool2d(kernel_size=2,stride=2),

            nn.Conv2d(32,64,kernel_size=3,padding='same'),
            nn.ReLU(),
            nn.BatchNorm2d(64),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64*7*7,128),
            nn.ReLU(),
            nn.Dropout(p=0.4),

            nn.Linear(128,64),
            nn.ReLU(),
            nn.Dropout(p=0.4),
            nn.Linear(64,10)
        )
    def forward(self,x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [16]:
learning_rate = 0.1
epochs = 50

In [18]:
# model
model = myCNN(1)
# loss function
criterion = nn.CrossEntropyLoss()
#optimizer 
optimizer = optim.SGD(model.parameters(),lr = learning_rate,weight_decay=1e-4)

In [19]:
# training loop

for epoch in range(epochs):
    total_epoch_loss = 0
    for batch_features , batch_labels in train_loader:
        # forward pass
        outputs = model(batch_features)
        # claculate loss
        loss = criterion(outputs,batch_labels)
        # clear grads
        optimizer.zero_grad()
        # back pass
        loss.backward()
        # update grads
        optimizer.step()

        total_epoch_loss = total_epoch_loss + loss.item()
    avg_loss = total_epoch_loss/len(train_loader)
    print(f'Epoch: {epoch+1},loss : {avg_loss}')

Epoch: 1,loss : 0.2614386424138987
Epoch: 2,loss : 0.1063079918108997
Epoch: 3,loss : 0.08248127906630301
Epoch: 4,loss : 0.059127336385288796
Epoch: 5,loss : 0.055779630161289485
Epoch: 6,loss : 0.04661885058689768
Epoch: 7,loss : 0.037740216504687396
Epoch: 8,loss : 0.035937418910380076
Epoch: 9,loss : 0.0324065665091689
Epoch: 10,loss : 0.028334843590291862
Epoch: 11,loss : 0.028606721565262432
Epoch: 12,loss : 0.023690986508942236
Epoch: 13,loss : 0.020200894864897366
Epoch: 14,loss : 0.020842135101488718
Epoch: 15,loss : 0.026286431403732574
Epoch: 16,loss : 0.02416856472379618
Epoch: 17,loss : 0.022639259517916836
Epoch: 18,loss : 0.02280520125769698
Epoch: 19,loss : 0.020281174644562517
Epoch: 20,loss : 0.020910381998323638
Epoch: 21,loss : 0.01794856105593241
Epoch: 22,loss : 0.01644179759753789
Epoch: 23,loss : 0.016235848889961168
Epoch: 24,loss : 0.014853891698557864
Epoch: 25,loss : 0.01508831884787522
Epoch: 26,loss : 0.015624718662304075
Epoch: 27,loss : 0.012631653149087

In [20]:
# set model to eval model
model.eval()

myCNN(
  (features): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (1): ReLU()
    (2): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same)
    (5): ReLU()
    (6): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=3136, out_features=128, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.4, inplace=False)
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): ReLU()
    (6): Dropout(p=0.4, inplace=False)
    (7): Linear(in_features=64, out_features=10, bias=True)
  )
)

In [21]:
# evaluation code
total  = 0 
correct = 0
with torch.no_grad():
    for batch_features , batch_labels in test_loader:
        outputs = model(batch_features)

        _,predicted = torch.max(outputs,1)
        
        total = total+batch_labels.shape[0]

        correct = correct + (predicted == batch_labels).sum().item()
    print(correct/total)

0.9918333333333333


In [22]:
# evaluation code over train loader
total  = 0 
correct = 0
with torch.no_grad():
    for batch_features , batch_labels in train_loader:
        outputs = model(batch_features)

        _,predicted = torch.max(outputs,1)
        
        total = total+batch_labels.shape[0]

        correct = correct + (predicted == batch_labels).sum().item()
    print(correct/total)

0.9998541666666667
